:meth:`SequenceFeature.prune_by_correlation` removes **empirically redundant** features: among features whose realized values are pairwise correlated beyond ``max_cor``, it keeps the one with the higher ``abs_auc`` and drops the others. The correlation is measured over the actual samples, making it complementary to CPP's in-run redundancy reduction (which compares scale vectors). Use it after :meth:`SequenceFeature.prune_by_variance` and before :meth:`TreeModel.select_features`.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
# gamma-secretase geometry: the TMD (~20 aa) comes from each protein's tmd_start/tmd_stop,
# flanked by short juxtamembrane domains of 4 residues each.
df_parts = sf.get_df_parts(df_seq=df_seq, jmd_n_len=4, jmd_c_len=4)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=50)
print(f"features from CPP.run: {len(df_feat)}")

features from CPP.run: 50


At ``max_cor=0.7`` no retained pair of features has absolute correlation above 0.7. The output is sorted by descending ``abs_auc`` and is deterministic across runs:

In [2]:
df_cor = sf.prune_by_correlation(df_feat=df_feat, df_parts=df_parts, max_cor=0.7)
print(f"kept {len(df_cor)} of {len(df_feat)} features")
df_cor.head()

kept 12 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-Pattern(C,2,5,8)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric point (Zimmerman et al., 1968)",0.402,0.130,0.130,0.080,0.088,0.000013,0.005011,"33,36,39"
2,"TMD_C_JMD_C-Segment(14,15)-AURR980119",Conformation,"α-helix (C-term, out)","α-helix (C-terminal, outside)",Normalized positional residue frequency at hel...,0.392,0.198,0.198,0.106,0.178,0.000022,0.005877,38
3,"TMD-Segment(10,11)-CHAM820102",Polarity,Hydrophobicity (interface),Free energy (interface),"Free energy of solution in water, kcal/mole (C...",0.389,0.169,-0.169,0.050,0.130,0.000026,0.006223,"27,28"
4,"TMD-Pattern(C,4,8,11)-FUKS010106",Composition,Membrane proteins (MPs),Proteins of mesophiles (INT),Interior composition of amino acids in intrace...,0.384,0.219,0.219,0.166,0.122,0.000033,0.006123,"20,23,27"


As with variance pruning, ``df_scales`` / ``accept_gaps`` / ``n_jobs`` configure the matrix build, and a pre-computed ``X`` (aligned column-for-column with ``df_feat``) can be supplied instead of ``df_parts``:

In [3]:
df_scales = aa.load_scales()
df_cor_full = sf.prune_by_correlation(df_feat=df_feat, df_parts=df_parts, df_scales=df_scales,
                                      max_cor=0.5, accept_gaps=True, n_jobs=1)
X = sf.feature_matrix(features=df_feat, df_parts=df_parts)
df_cor_X = sf.prune_by_correlation(df_feat=df_feat, X=X, max_cor=0.5)
print(f"via df_parts+params: {len(df_cor_full)} | via pre-computed X: {len(df_cor_X)}")

via df_parts+params: 3 | via pre-computed X: 3


The two pruners compose, and the result drops straight into model-based selection (:meth:`TreeModel.select_features`):

In [4]:
df_pruned = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, threshold=0.0)
df_pruned = sf.prune_by_correlation(df_feat=df_pruned, df_parts=df_parts, max_cor=0.5)
print(f"variance -> correlation kept {len(df_pruned)} of {len(df_feat)} features")
df_pruned.head()

variance -> correlation kept 3 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-Pattern(C,2,5,8)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric point (Zimmerman et al., 1968)",0.402,0.130,0.130,0.080,0.088,0.000013,0.005011,"33,36,39"
2,"TMD_C_JMD_C-Pattern(C,2,6,9)-GEIM800103",Conformation,Unclassified (Conformation),α-helix (β-proteins),Alpha-helix indices for beta-proteins (Geisow-...,0.359,0.163,-0.163,0.137,0.080,0.000104,0.005900,"32,35,39"


**What can go wrong?** A ``df_feat`` with fewer than two features has nothing to compare and is returned unchanged:

In [5]:
one = df_feat.head(1)
print(f"single-feature input -> {len(sf.prune_by_correlation(df_feat=one, df_parts=df_parts))} feature")

single-feature input -> 1 feature
